In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd


read gold table for clustering

In [0]:
df_spark = spark.table("gold_customer_kmeans_features")
df_spark.printSchema()

convert to pandas-> required for sklearn

In [0]:
df = df_spark.toPandas()

select features for clustering

In [0]:
feature_cols = [
    "Age",
    "Membership_Years",
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session",
    "Total_Purchases",
    "Average_Order_Value",
    "Lifetime_Value"
]


In [0]:
df[feature_cols].head()

feature scaling

In [0]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

In [0]:
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

# Show first 5 rows of scaled features
X_scaled_df.head()

comparison of feature values and scaled feature values

In [0]:
comparison_df = pd.concat(
    [
        df[feature_cols].head(5).reset_index(drop=True),
        X_scaled_df.head(5).reset_index(drop=True)
    ],
    axis=1,
    keys=["Original_Features", "Scaled_Features"]
)

comparison_df


In [0]:
df[feature_cols].describe()

In [0]:
X_scaled_df.describe()

applying k-means clustering

In [0]:
kmeans = KMeans(n_clusters=4, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

In [0]:
df.groupby("Cluster")[[
    "Lifetime_Value",
    "Total_Purchases",
    "Login_Frequency",
    "Average_Order_Value",
    "Churned"
]].mean()

assigning meaningful cluster names

In [0]:
def assign_cluster_name(cluster):
    if cluster == 0:
        return "High Value Loyal Customers"
    elif cluster == 1:
        return "Discount‑Driven / Medium Value"
    elif cluster == 2:
        return "High Engagement Explorers"
    else:
        return "Low Value / Low Engagement"

df["Cluster_Name"] = df["Cluster"].apply(assign_cluster_name)

In [0]:
final_clustered_df = spark.createDataFrame(df)

save the dataframe as a table

In [0]:
final_clustered_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("clustered_customers_kmeans")

validation check

In [0]:
%sql
SELECT
  Cluster_Name,
  COUNT(*) AS customers
FROM clustered_customers_kmeans
GROUP BY Cluster_Name;